# Arabic OCR — QLoRA fine-tuning on Colab (crash-safe)

Fine-tunes a Qwen2-VL-family OCR model on a reserved training split, merges
the adapter, registers it, and scores it on the **untouched** benchmark test
split via the normal `bench run` path.

**Built for Colab disconnects.** Checkpoints and the base-model cache live on
**Google Drive**; every checkpoint is also mirrored to a **private Hugging
Face repo**. If the runtime dies, just reopen the notebook and **Run all** —
every cell is idempotent and training continues from the last checkpoint.
Model *weights* are never committed to git (too big); only the small
**results** JSON/CSV/HTML are pushed back to the repo.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`; add Colab
secrets `GH_TOKEN` (repo push) and `HF_TOKEN` (Muharaf download + checkpoint
backup). ~5–15 min for `smoke.yaml`.

## 1. GPU

In [ ]:
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print('GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=memory.total,memory.used --format=csv
else:
    print('NO GPU — will run on CPU (much slower; big 4-bit models are skipped).')
    print('For a real run: Runtime -> Change runtime type -> T4 GPU.')

## 2. Mount Google Drive — the durable store

Checkpoints, the merged model, and the HF model cache all go here so they
survive a runtime recycle.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
WORK = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench')
(WORK / 'training').mkdir(parents=True, exist_ok=True)
(WORK / 'hf_cache').mkdir(parents=True, exist_ok=True)
print('durable work dir:', WORK)

## 3. Persistent caches + auth

`HF_HOME` points at Drive so the ~5 GB base model is downloaded **once**, not
again after every disconnect. Tokens come from Colab secrets.

In [ ]:
import os
os.environ['HF_HOME'] = str(WORK / 'hf_cache')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from google.colab import userdata
for k in ('HF_TOKEN', 'GH_TOKEN'):
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v
    except Exception as e:
        print(f'no {k} secret:', e)
if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF login ok')

## 4. Clone the repo

In [ ]:
# Bulletproof checkout: works whether the repo is missing, at the wrong
# path, or has local edits. Always ends up at the latest origin/main.
import os
REPO = '/content/arabic-ocr-benchmark'
# Copy Colab secrets into the environment. SWEEP_RUNNER_ID pins a STABLE
# name for this account (e.g. 'runner-a') so a restarted runner re-adopts
# its own in-flight batches instantly instead of waiting out the lease TTL.
for _k in ('GH_TOKEN', 'HF_TOKEN', 'SWEEP_RUNNER_ID'):
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v: os.environ[_k] = _v
    except Exception:
        pass
token = os.environ.get('GH_TOKEN')
if not token:
    print('no GH_TOKEN secret — clone/push will be anonymous / read-only')
import subprocess, time, sys
url = f"https://{(token + '@') if token else ''}github.com/rabeeh97/arabic-ocr-benchmark.git"
def _sh(*a):
    return subprocess.run(a, capture_output=True, text=True)
_updated = False
if os.path.isdir(REPO + '/.git'):
    r = _sh('git', '-C', REPO, 'fetch', '--depth', '1', 'origin', 'main')
    if r.returncode == 0:
        _sh('git', '-C', REPO, 'reset', '--hard', 'origin/main')
        # drop untracked leftovers (e.g. a stale sweep_manifest.json); keeps gitignored _cache
        _sh('git', '-C', REPO, 'clean', '-fd', '--', 'results')
        _updated = True
    else:
        print('existing checkout is stale and fetch failed — re-cloning\n', r.stderr[-300:])
        _sh('rm', '-rf', REPO)
if not _updated:
    for _try in range(1, 5):
        _sh('rm', '-rf', REPO)
        r = _sh('git', 'clone', '--depth', '1', url, REPO)
        if r.returncode == 0 and os.path.isdir(REPO + '/.git'):
            break
        print(f'clone attempt {_try}/4 failed:\n' + (r.stderr[-400:] or r.stdout[-400:]))
        time.sleep(5 * _try)
if not os.path.isdir(REPO + '/.git'):
    raise RuntimeError(
        'git clone failed. Checklist: (1) add a Colab secret named GH_TOKEN — a '
        'fine-grained PAT with Contents: read on rabeeh97/arabic-ocr-benchmark '
        '(the repo is private); (2) enable notebook access to that secret in the '
        'left sidebar key icon; (3) if it is a network blip, just re-run this cell.'
    )
os.chdir(REPO)
# src-layout package: importable straight from the checkout even if the
# `pip install -e` cell fails or is skipped (this is what caused the old
# 'No module named arabic_ocr_bench' in the sweep cell).
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
print(_sh('git', 'log', '--oneline', '-3').stdout)

## 5. Install — core + local + train extras + Arabic fonts

In [ ]:
!pip install -q -e '.[local,train]'
# transformers>=4.49, peft, trl, bitsandbytes, qwen-vl-utils, imagehash, fonttools
!pip uninstall -y -q torchao 2>/dev/null  # stale Colab torchao breaks model load
!python scripts/fetch_fonts.py   # extra Arabic faces for synthetic rendering (no-op if a config has no synthetic source)

## 6. Load the experiment config + point it at Drive

`smoke.yaml` proves the pipeline. Real experiments: one file per run under
`configs/training/` (see `docs/training-plan.md`).

In [ ]:
TRAIN_CFG = 'configs/training/smoke.yaml'
PUSH_TO_HUB = True   # mirror every checkpoint to a private HF repo (needs HF_TOKEN)

from arabic_ocr_bench.training.config import load_training_config
cfg = load_training_config(TRAIN_CFG)   # also runs the no-scored-split guard

# redirect all heavy output to the durable Drive dir
cfg.output_dir = str(WORK / 'training' / cfg.experiment)

if PUSH_TO_HUB and os.environ.get('HF_TOKEN'):
    from huggingface_hub import whoami
    who = whoami()['name']
    cfg.train.push_to_hub = True
    cfg.train.hub_model_id = f'{who}/arabic-ocr-{cfg.experiment}'
    cfg.train.hub_private = True
    print('checkpoint backup ->', cfg.train.hub_model_id)
else:
    cfg.train.push_to_hub = False

print(cfg.experiment, '|', cfg.method, '|', cfg.base_model.hf_id)
print('output_dir :', cfg.output_dir)
print('eval on    :', cfg.eval_datasets)

## 6b. Reuse notebook 06's synthetic render

For a `synthetic_printed` config, point the source at the **same render
cache `06_colab_prepare_data.ipynb` wrote** &mdash; a Drive folder named by a
hash of the render parameters, so 05 and 06 (and the 5k / 15k configs)
all share it. Lines 06 already rendered are restored from `images.tar`;
only the tail is rendered here. No-op if the config has no synthetic source.

In [ ]:
import pathlib
from arabic_ocr_bench.training.synth import synth_render_key
_synth = [s for s in cfg.data.sources if 'synth' in s.adapter.lower()]
if _synth:
    _rkey = synth_render_key(_synth[0].params)
    _drive = WORK / 'synth' / f'render-{_rkey}'          # Drive: images.tar + pool cache
    _work  = pathlib.Path('/content/synth') / f'render-{_rkey}'   # fast local render
    _drive.mkdir(parents=True, exist_ok=True)
    for s in _synth:
        s.params['cache_dir'] = str(_drive)
        s.params['work_dir'] = str(_work)
        s.params['flush_every'] = 200
    _tar = _drive / 'images.tar'
    if _tar.exists():
        import tarfile
        with tarfile.open(_tar) as _tf:
            print('synth render cache <-', _drive, '(', len(_tf.getnames()), 'cached lines)')
    else:
        print('synth render cache <-', _drive, '(empty — will render)')
else:
    print('no synthetic source in this config')

## 7. Resume status

Shows whether a checkpoint from a previous (interrupted) run is on Drive.
If so, the training cell picks up from there automatically.

In [ ]:
from arabic_ocr_bench.training.checkpoints import latest_checkpoint
ckpt = latest_checkpoint(cfg.output_dir)
print('RESUME from', ckpt) if ckpt else print('fresh start (no checkpoint on Drive yet)')

## 8. Contamination check (blocking)

Perceptual-hash + normalized-text overlap of every training source against
every eval test split. Writes `results/contamination_report.json`.

In [ ]:
from arabic_ocr_bench.training.contamination import check_datasets
specs = [{**s.model_dump(exclude_none=True), 'name': s.name} for s in cfg.data.sources]
rep = check_datasets(specs, cfg.eval_datasets, limit=None)
for p in rep['pairs']:
    print(p['train_source'], 'vs', p['test_dataset'], '->',
          p['n_text_collisions'], 'text,', p['n_near_dup_images'], 'image')
assert rep['n_flagged'] == 0, 'contamination flagged — see results/contamination_report.json'

## 9. Peek at the assembled data

In [ ]:
from arabic_ocr_bench.training.data import assemble_dataset
bundle = assemble_dataset(cfg.data, seed=cfg.train.seed)
print('train', len(bundle['train']), ' val', len(bundle['val']))
row = bundle['train'][0]
display(row['images'][0]); print('target:', row['messages'][2]['content'][0]['text'])

## 10. Train — resumable

Writes a checkpoint to Drive every epoch (or every `save_steps` steps if the
config sets `save_strategy: steps`) and mirrors it to the HF repo. **If Colab
drops: reopen, Run all — cells 2–10 are idempotent and this resumes from the
last checkpoint.**

In [ ]:
from arabic_ocr_bench.training.sft_vlm import train
adapter_dir = train(cfg)   # -> <Drive>/training/<experiment>/
print('adapter ->', adapter_dir)

## 11. Merge + register

Merge the adapter into base weights (also on Drive) and add an `hf_local` row
to `configs/models.yaml`. If pushed to the Hub, the row points at the HF repo
id (reproducible anywhere); otherwise at the Drive path (this session only).

In [ ]:
from arabic_ocr_bench.training.merge import merge_adapter
from arabic_ocr_bench.training.registry import build_model_entry, register_model
merged = merge_adapter(adapter_dir, str(WORK / 'training' / f'{cfg.experiment}-merged'))
model_name = f'{cfg.experiment}-ft'
hf_id = cfg.train.hub_model_id if cfg.train.push_to_hub else str(merged)
register_model(model_name, build_model_entry(
    model_name, hf_id, arch=cfg.base_model.arch, adapter_id=None,
    card=f'{cfg.experiment}: {cfg.method} on {[s.name for s in cfg.data.sources]}'))
print('registered', model_name, '->', hf_id)

## 12. Score on the untouched test split

In [ ]:
from arabic_ocr_bench.runner import run
from arabic_ocr_bench.report import score_run
run_cfg = {
    'models': [model_name],
    'datasets': cfg.eval_datasets,
    'limit': 50,   # None = whole test split
    'prompt_version': cfg.data.prompt_version,
    'normalization_profiles': ['raw', 'light', 'diac_insensitive'],
    'concurrency': 1, 'results_dir': 'results',
}
run_dir = run(run_cfg)
print(open(score_run(run_dir, profiles=tuple(run_cfg['normalization_profiles']))).read())

## 13. Leaderboard

In [ ]:
# Build the shareable HTML leaderboard from every run under results/
!python scripts/make_leaderboard.py results -o results/leaderboard.html
from IPython.display import HTML
HTML(open('results/leaderboard.html').read())

## 14. Push **results** back to the repo

Commits `results/<run_id>/`, the leaderboard, and the contamination report —
small text/JSON. Weights are **not** here: they are on Drive and (if enabled)
the HF repo. Needs `GH_TOKEN`.

In [ ]:
import subprocess
tok = os.environ.get('GH_TOKEN')
if tok:
    subprocess.run(['git','config','user.email','colab@arabic-ocr-bench'], check=True)
    subprocess.run(['git','config','user.name','colab-trainer'], check=True)
    add = ['git','add','-f', f'results/{run_dir.name}',
           'results/leaderboard.html','results/leaderboard.json',
           'results/contamination_report.json', 'configs/models.yaml']
    subprocess.run(add, check=True)
    msg = f'train: {model_name} on {cfg.eval_datasets} (cfg={cfg.experiment})'
    r = subprocess.run(['git','commit','-m',msg], capture_output=True, text=True)
    print(r.stdout or r.stderr)
    p = subprocess.run(['git','push','origin','HEAD:main'], capture_output=True, text=True)
    print(p.stdout or p.stderr)
else:
    print('no GH_TOKEN — results are on Drive only')

## 15. Copy the run dir to Drive too (belt & braces)

In [ ]:
import shutil
dst = WORK / 'results' / run_dir.name
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(run_dir, dst, dirs_exist_ok=True)
print('saved ->', dst)

## Recovering from a disconnect

1. Reopen this notebook in Colab, T4 runtime.
2. **Runtime -> Run all.** Cells 2–10 re-mount Drive, re-clone, re-install
   (fast, cached), reload the config, and `train()` resumes from the last
   `checkpoint-*` on Drive. Nothing is recomputed.
3. If Drive somehow lost the checkpoint, the HF repo
   `cfg.train.hub_model_id` still has every pushed checkpoint — download the
   latest into `cfg.output_dir` and Run all.

## Next

- Log the run in `docs/experiments/<date>-<name>.md` (template there).
- Real experiments: `docs/training-plan.md` (milestones M1–M4). For long runs
  set `train.save_strategy: steps` + `train.save_steps` in the config so a
  checkpoint lands well before the ~12 h Colab cap.
- Data prep / vetting: `06_colab_prepare_data.ipynb`.